<a href="https://colab.research.google.com/github/yukinaga/minnano_kaggle/blob/main/section_4/02_automl_houseprices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mac では動かない

# 住宅価格の予測
AutoMLにより、住宅の価格を予測します。  
訓練済みのモデルによる予測結果は、csvファイルに保存して提出します。  

## PyCaretのインストール
AutoMLをサポートするライブラリ、PyCaretをバージョンを指定してインストールします。

In [1]:
!pip install numpy==1.21.4 numba==0.53
!pip install pycaret
!pip install pandas-profiling==3.1.0

ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement numpy==1.21.4 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17.0, 1.17.1, 1.17.2, 1.17.3, 1.17.4, 1.17.5, 1.18.0, 1.18.1, 1.18.2, 1.18.3, 1.18.4, 1.18.5, 1.19.0, 1.19.1, 1.19.2, 1.19.3, 1.19.4, 1.19.5, 1.20.0, 1.20.1, 1.20.2, 1.20.3, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.22.2, 

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.2/261.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.4/102.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 25.8 MB/s eta 0:00:00
  Created wheel for htmlmin: filename=htmlmin-0.1.12-py3-none-any.whl size=27081 sha256=c545565c8be101c43da928d29bd092c72f220ee64092c68128c91c3497f3a264
  Stored in directory: /root/.cache/pip/wheels/5f/d4/d7/4189b07b5902ee9f3ce0dbb14909fbe8037c39d6c63ffd49c9
  Created wheel for markupsafe: filename=MarkupSafe-2.0.1-cp312-cp312-linux_x86_64.whl size=28149 sha256=5352ef14bfa394b520cc81da4544e84da36047a6ce37c6c927b8b55cb24b1164


## Google Colabの設定
Google Colab環境でPyCaretのインタラクティブな要素を表示するためには、以下のコードを実行する必要があります。

In [ ]:
# from pycaret.utils import enable_colab

# enable_colab()

## データの読み込み
以下のページから住宅価格のデータをダウロードして、「train.csv」「test.csv」をノートブック環境にアップします。  
https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data

以下のコードは、これらのデータを読み込みます。   
訓練データには住宅価格を表す"SalePrice"の列がありますが、テストデータにはありません。  
訓練済みのモデルに、テストデータを入力して判定した結果を提出することになります。  


In [ ]:
import numpy as np
import pandas as pd

train_data = pd.read_csv("train.csv")  # 訓練データ
test_data = pd.read_csv("test.csv") # テストデータ

train_data.head()

## 環境の設定
PyCaretの環境を設定します。  
setup関数はPyCaretの環境を初期化しますが、PyCaretの他の関数を実行する前に呼び出す必要があります。      
データの型に問題が無ければ、空白を入力することで設定を完了することができます。  
  
今回のデータには一部欠損があるので、欠損データに対する対応を設定する必要があります。  
`numeric_imputation="mean"`により数値データの欠損には平均値があてがわれ、`categorical_imputation="mode"`によりカテゴリデータの欠損には最頻値があてがわれます。  

https://pycaret.readthedocs.io/en/latest/api/regression.html#pycaret.regression.setup


In [ ]:
from pycaret.regression import setup

clf = setup(data=train_data, target="SalePrice", session_id=123,  # 環境の初期化
            numeric_imputation="mean", categorical_imputation="mode")

## モデルの比較
様々なモデルを比較して、性能を評価します。  
compare_models関数は、ライブラリ内のすべてのモデルを使って訓練を行い、スコアを評価します。  

In [ ]:
from pycaret.regression import compare_models

best_model = compare_models()  # 全てのモデルを訓練し、評価する

わずか1行のコードで、15以上の機械学習モデルを訓練し、評価することができました。  
  
最もスコアの良いモデルの概要を表示します。


In [ ]:
print(best_model)

## モデルの作成
models関数により、全ての使用可能な機械学習モデルを確認することができます。

In [ ]:
from pycaret.regression import models

models()  # 機械学習モデルの一覧

create_model関数は、「交差検証」を用いて個別のモデルの訓練と評価を行います。  
今回は、先程最も精度が高かった「勾配ブースティング」による回帰（Gradient Boosting Regressor）のモデルを作成します。  


In [ ]:
from pycaret.regression import create_model

gbr = create_model("gbr")  # 勾配ブースティングのモデルを作成

訓練済みモデルの概要を表示します。  

In [ ]:
print(gbr)

## ハイパーパラメータの調整
tune_model関数を使用し、ハイパーパラメータを最適化します。

In [ ]:
from pycaret.regression import tune_model

tuned_gbr = tune_model(gbr)  # ハイパーパラメータの調整

ハイパーパラメータを調整済みの、モデルの概要を表示します。  

In [ ]:
print(tuned_gbr)

## モデルを評価する
plot_model関数を使い、各特徴量の重要度をプロットします。

In [ ]:
from pycaret.regression import plot_model

plot_model(tuned_gbr, plot="feature")  # 各特徴の重要度をプロット

## 本番用のモデルを作成
`finalize_model`関数により全ての訓練データを使ってモデルを訓練し、本番用のモデルを作成します。   

In [ ]:
from pycaret.regression import finalize_model

final_gbr = finalize_model(tuned_gbr)
print(final_gbr)

## 提出用のデータを作成
テストデータを使って予測を行います。  
予測結果には、予測値を表す「Label」列が含まれます。

In [ ]:
from pycaret.regression import predict_model

test_pred = predict_model(final_gbr, data=test_data)  # 予測
test_pred.head()

形式を整えた上で、Kaggleに提出するためのcsvファイルを保存します。

In [ ]:
# 形式を整える
subm_data = test_pred[["Id", "prediction_label"]]  # 列を抜き出す
subm_data = subm_data.rename(columns={"Label" : "SalePrice"})  # 列名の変更

# 提出用のcsvファイルを保存
subm_data.to_csv("submission_houseprices.csv", index=False)

subm_data